# Figure G01: the protein × 3Di entropy landscape of six-frame ORFs, and the log₂k ceiling (GTDB r232)

**Status:** draft, Gadi. The ID is provisional.

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figG01_joint_sample.tsv.gz`, `data/figG01_marginals_sample.tsv.gz` | Gadi `/g/data/ob80/re3494/gtdb_entropy/figure_samples/sample_{bac,arc}.tsv.gz` | `Validation/gtdb/scripts/08b_sample_for_figures.pbs` |
| `data/figG01_population_bands.tsv.gz` | Gadi `population_summary/population_entropy_{bac,arc}_bands.tsv` | `29_population_entropy_summary.pbs` (genome_entropy #100) |

- **Source:** `linsalrob/genome_entropy` `main` at `97a792f`.
- **Reduction:** `manuscript/prep/gtdb_prepare_data.py --only G01`. It bins the sample into a 2-D histogram (0.02 bits) and 1-D histograms (3Di 0.005 bits, protein 0.01 bits). The population band fractions are copied verbatim.
- **Sample:** systematic, every 300th bacterial and every 30th archaeal ORF row. It is restricted to genomes with ≥1 deposited CDS, as in every report figure. That leaves:
  - bacteria: 4,893,192 ORFs (1,028,059 matched);
  - archaea: 1,138,714 ORFs (315,638 matched).

  The sample reproduces the population's share of unmatched ORFs below log₂3 to four decimal places (0.8760 and 0.8409).
- **Definitions:**
  - **Matched** means the ORF matches a GenBank CDS (`in_genbank`).
  - **Entropies** are plug-in Shannon entropies in bits of the translated ORF (protein, 21 symbols including X) and of its ModernProst 3Di encoding (20 states).
  - **Binning:** 3Di bins are right-closed, (lo, hi], so an ORF exactly at log₂k sits immediately below the ceiling line. 0.04% of rows fall outside the 2-D protein window (2.35–4.35 bits). They are counted in `n_outside_joint_grid` and are not drawn in panels a–d. The marginals keep every row in [0, 4.4] bits.
- **Densities:** each panel is normalised within its own class, so it compares shapes, not abundance.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}

import math
from matplotlib.colors import LogNorm

CLASS = {"matched": "#0072B2", "unmatched": "#E69F00"}
CEILINGS = [(0.0, "log$_2$1"), (1.0, "log$_2$2"), (math.log2(3), "log$_2$3"), (2.0, "log$_2$4")]


def ceilings(ax, axis="y", label=False):
    for v, t in CEILINGS:
        (ax.axhline if axis == "y" else ax.axvline)(
            v, color=MUTED, lw=0.6, ls=(0, (1, 2)), zorder=3)
        if label:
            ax.text(v, 1.0, t, transform=ax.get_xaxis_transform(), rotation=90,
                    ha="right", va="top", fontsize=5.8, color=MUTED)


In [ ]:
joint = pd.read_csv(DATA / "figG01_joint_sample.tsv.gz", sep="\t")
marg = pd.read_csv(DATA / "figG01_marginals_sample.tsv.gz", sep="\t")
bands = pd.read_csv(DATA / "figG01_population_bands.tsv.gz", sep="\t")
GP = np.round(np.arange(2.35, 4.35 + 1e-9, 0.02), 4)
GT = np.round(np.arange(-0.015, 4.40 + 1e-9, 0.02), 4)

fig = plt.figure(figsize=(7.2, 5.4))
gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 1.05], hspace=0.42)
top = gs[0].subgridspec(1, 5, width_ratios=[1, 1, 1, 1, 0.06], wspace=0.12)
bot = gs[1].subgridspec(1, 3, width_ratios=[1.35, 1.0, 0.85], wspace=0.38)

norm = LogNorm(1e-6, 1e-2)
cells = [("bacteria", "matched"), ("bacteria", "unmatched"),
         ("archaea", "matched"), ("archaea", "unmatched")]
for k, (dom, cls) in enumerate(cells):
    ax = fig.add_subplot(top[k])
    g = joint[(joint.domain == dom) & (joint.cls == cls)]
    H = np.zeros((len(GP) - 1, len(GT) - 1))
    H[np.searchsorted(GP, g.protein_lo.round(4)), np.searchsorted(GT, g.three_di_lo.round(4))] = g.n
    n_cls = marg[(marg.domain == dom) & (marg.cls == cls)].n_class.iloc[0]
    F = np.where(H > 0, H / n_cls, np.nan)
    mesh = ax.pcolormesh(GP, GT, F.T, norm=norm, cmap="viridis", rasterized=True)
    ceilings(ax, "y")
    ax.set_xlim(2.35, 4.35)
    ax.set_ylim(-0.015, 4.3)
    ax.set_xticks([2.5, 3.0, 3.5, 4.0])
    ax.set_xlabel("protein entropy (bits)")
    if k == 0:
        ax.set_ylabel("3Di entropy (bits)")
        ax.text(-0.32, 1.04, "a", transform=ax.transAxes, fontsize=10,
                fontweight="bold", ha="right", va="bottom")
    else:
        ax.set_yticklabels([])
    ax.set_title(f"{dom}, {cls}\nn = {n_cls:,}", loc="left", fontsize=7,
                 color=CLASS[cls], pad=4)
cax = fig.add_subplot(top[4])
cb = fig.colorbar(mesh, cax=cax)
cb.set_label("fraction of class per cell", fontsize=6.5)
cb.ax.tick_params(labelsize=6)

def density(dom, cls, alphabet):
    g = marg[(marg.domain == dom) & (marg.cls == cls) & (marg.alphabet == alphabet)]
    w = (g.bin_hi - g.bin_lo).to_numpy()
    return np.append(g.bin_lo.to_numpy(), g.bin_hi.iloc[-1]), g.n.to_numpy() / (g.n_class.iloc[0] * w)

for j, (alphabet, xlabel, letter) in enumerate((("three_di", "3Di entropy (bits)", "b"),
                                                ("protein", "protein entropy (bits)", "c"))):
    ax = fig.add_subplot(bot[j])
    for dom, ls in (("bacteria", "-"), ("archaea", (0, (2.5, 1.2)))):
        for cls in ("unmatched", "matched"):
            edges, dens = density(dom, cls, alphabet)
            ax.stairs(dens, edges, color=CLASS[cls], lw=0.8, ls=ls,
                      label=f"{dom}, {cls}")
    if alphabet == "three_di":
        ceilings(ax, "x")
        ax.set_xlim(-0.05, 4.4)
        ax.set_ylim(0, 2.6)
        ax.annotate("log$_2$3 = 1.585", xy=(math.log2(3), 1.2), xytext=(2.05, 1.55),
                    fontsize=6, color=INK, arrowprops=dict(arrowstyle="-", lw=0.5, color=MUTED))
        ax.text(2.0, 2.5, "spike at 0 (single-state\nencodings) runs off scale",
                fontsize=5.8, color=MUTED, va="top")
    else:
        ax.set_xlim(2.35, 4.35)
        ax.set_ylim(0, 4.9)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("density (within class)")
    panel(ax, letter)
    if j == 1:
        ax.legend(frameon=False, fontsize=6, loc="upper left", handlelength=2.2)

ax = fig.add_subplot(bot[2])
b = bands[bands.metric == "three_di"].copy()
b["cls"] = b.group.str.replace("CDS-bearing genomes: ", "", regex=False)
w = 0.38
for k, dom in enumerate(("bacteria", "archaea")):
    for i, cls in enumerate(("matched", "unmatched")):
        r = b[(b.domain_name == dom) & (b.cls == cls)].iloc[0]
        v = r["frac_lt_1.5850"] * 100
        x = i + (k - 0.5) * w
        ax.bar(x, v, width=w, color=CLASS[cls], alpha=1.0 if dom == "bacteria" else 0.55,
               edgecolor=CLASS[cls], lw=0.6)
        if v > 20:
            ax.text(x, v - 3, f"{v:.1f}", ha="center", va="top", fontsize=6,
                    color="white", fontweight="bold")
        else:
            ax.text(x, v + 2, f"{v:.1f}", ha="center", fontsize=6, color=INK)
ax.set_xticks([0, 1], ["matched", "unmatched"])
ax.set_ylabel("% of ORFs with 3Di entropy < log$_2$3")
ax.set_ylim(0, 105)
ax.text(0.03, 0.72, "solid: bacteria\nfaded: archaea\nfull population,\nno sampling",
        transform=ax.transAxes, va="top", fontsize=5.8, color=MUTED)
panel(ax, "d")

save(fig, "figG01_entropy_landscape")


In [ ]:
b3 = bands[bands.metric == "three_di"]
print(b3[["domain_name", "group", "n_valid", "frac_lt_1.5850", "frac_lt_2.5000"]].to_string(index=False))
for (d, c), g in marg[marg.alphabet == "three_di"].groupby(["domain", "cls"]):
    print(d, c, "sample n", g.n_class.iloc[0], "below log2(3)", round(g[g.bin_hi <= 1.585].n.sum() / g.n.sum(), 4),
          "| outside 2-D window", g.n_outside_joint_grid.iloc[0])


## Caption (draft)

**Figure G01. Unannotated six-frame ORFs separate from annotated genes in 3Di entropy, not protein entropy, and most of them sit under the log₂3 ceiling.**

**(a)** The joint distribution of protein entropy against 3Di entropy for ORFs in GTDB r232 genomes that carry at least one deposited CDS, split by whether the ORF matches a GenBank CDS. Each panel is normalised within its class, and the colour scale is logarithmic. The data are a systematic sample: every 300th bacterial and every 30th archaeal ORF row, 4.89 M and 1.14 M ORFs.

**(b, c)** Marginal 3Di and protein entropy densities by class and domain. 3Di bins are 0.005 bits wide and closed on the right, so the edge at log₂3 = 1.585 is shown unsmoothed. Dotted lines mark log₂k for k = 1–4, the most a string using k distinct states can score. The edge at 1.585 is that arithmetic ceiling, not a biological threshold: it falls in the same place in both domains.

**(d)** The share of all ORFs below log₂3 in CDS-bearing genomes, computed over the full population with no sampling: 1.47 × 10⁹ bacterial and 3.4 × 10⁷ archaeal ORFs. In bacteria the share is 87.6% of unmatched ORFs against 3.3% of matched ORFs; in archaea it is 84.1% against 3.7%.

Over the full population, mean 3Di entropy differs between matched and unmatched ORFs by 2.08 bits (bacteria) and 1.99 bits (archaea). Mean protein entropy differs by only 0.39 and 0.37 bits.
